<a href="https://colab.research.google.com/github/JuPaRozo/Programaci-n-en-phyton/blob/main/clase_6/Funciones_Python_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Curso de Programación de Computadores en Python
## Variable scope, funciones como parámetros y funciones anidadas
#### Universidad Nacional de Colombia

---
### Objetivos
- Entender el **alcance (scope)** de variables en Python y el uso de `global` y `nonlocal`.
- Usar **funciones como parámetros** (higher-order functions) y ver casos prácticos.
- Crear y usar **funciones anidadas** y **closures** para mantener estado.
- Realizar ejercicios prácticos en cada sección dentro del notebook.


## 🔹 Cómo usar este notebook

- Ejecuta las celdas de código con `Shift+Enter` para ver ejemplos.
- Intenta resolver los ejercicios (`Esqueleto`).

# 1️⃣ Variable scope y variables globales

Python sigue la regla **LEGB** para resolver nombres:
- **L**ocal — nombres en la función actual.
- **E**nclosing — nombres en funciones que envuelven (closures).
- **G**lobal — nombres en el módulo.
- **B**uiltins — nombres incorporados (`len`, `print`, ...).

Entender el scope es clave para evitar errores como `UnboundLocalError` y para decidir si usar `global` o `nonlocal`.

In [ ]:
# Ejemplo 1: variable local y global con el mismo nombre
x = 10  # variable global
def mostrar_x_local():
    x = 5  # variable local, oculta la global dentro de la función
    print('x dentro de la función =', x)

def mostrar_x_global():
    print('x global =', x)

mostrar_x_local()
mostrar_x_global()
print('x fuera de funciones =', x)

x dentro de la función = 5
x global = 10
x fuera de funciones = 10


### Ejemplo 2: intento de modificar variable global sin `global` — causa UnboundLocalError

Si intentas asignar a `x` dentro de la función, Python crea una variable local y si antes de la asignación intentas usarla, obtendrás `UnboundLocalError`.

In [9]:
y = 100
def fallar_modificacion():
    # la siguiente línea provoca UnboundLocalError si descomentamos el print antes de asignar
    # print(y)  # UnboundLocalError si la función asigna a y luego intenta leer antes de asignar
    y = 1  # Python interpreta y como variable local -> error
    return(y)

# Para ver el error, descomenta la llamada (advertencia: lanzará una excepción)
# fallar_modificacion()
fallar_modificacion()

1

### Cómo modificar la variable global: usar `global` (con cautela)
Usar `global` permite asignar a una variable definida en el ámbito global desde dentro de una función. No es recomendable abusar de `global` en código mantenible; mejor retornar valores y reasignar externamente cuando sea posible.

In [ ]:
contador = 0
def incrementar_global():
    global contador
    contador += 1

print('contador inicial =', contador)
incrementar_global()
print('contador después =', contador)

contador inicial = 0
contador después = 1


### 📝 Ejercicio 1 — Scope y `global` (Esqueleto)

Tienes el siguiente código. Completa la función `usar_global` para aumentar la variable `valor_global` en la cantidad `inc`. Decide si usar `global` o, preferiblemente, retorna el nuevo valor.

Completa el esqueleto y prueba ambas estrategias (usar `global` y la versión que devuelve el valor).

In [20]:
# Esqueleto
x = int(input())
valor_global = 50
def usar_global(inc):
    global valor_global
    valor_global += inc
    return(valor_global)
usar_global(x)

10


60

### `nonlocal` — modificar variable del scope envolvente (en closures)

Cuando tienes funciones anidadas, `nonlocal` permite modificar la variable del scope envolvente (pero no global). Es útil en closures que mantienen estado.

In [ ]:
def contador_factory():
    contador = 0
    def incrementar():
        nonlocal contador
        contador += 1
        return contador
    return incrementar

c = contador_factory()
print(c())
print(c())
print(c())

1
2
3


### 📝 Ejercicio 2 — `nonlocal` (Esqueleto)
Implementa `make_accumulator(initial)` that returns a function that adds a given value to the accumulator and returns the total (use `nonlocal`).

In [24]:
# Esqueleto
def make_accumulator(initial):
    # devuelve una función 'add' que toma un número y lo suma al acumulador
    def add_accumulator(number):
        nonlocal initial
        initial += number
        return initial
    return add_accumulator

# Prueba sugerida:
acc = make_accumulator(10)
print(acc(5))  # 15
print(acc(3))  # 18

15
18


# 2️⃣ Functions as parameters (Funciones como parámetros)

Python permite pasar funciones como argumentos, retornarlas y asignarlas a variables. Esto habilita patrones poderosos: callbacks, mappers, filtros, funciones de orden superior.

Ventajas:
- Código más abstracto y reutilizable.
- Facilita composición de comportamientos.
- Base de patrones como `map`, `filter`, `reduce` y decoradores.

In [ ]:
# Ejemplo: aplicar una función dos veces
def aplicar_dos_veces(func, valor):
    return func(func(valor))

def sumar_dos(x):
    return x + 2

print(aplicar_dos_veces(sumar_dos, 3))  # 7

7


### Ejemplo: funciones como parámetros en sorting
La función `sorted()` acepta `key=` que es otra función que se aplica a cada elemento para decidir el orden.

In [ ]:
nombres = ['Ana', 'José', 'María', 'Alberto']
print(sorted(nombres, key=lambda s: len(s)))  # ordena por longitud

### 📝 Ejercicio 3 — Funciones como parámetros (Esqueleto)
Implementa `filtrar_y_aplicar(func_cond, func_accion, start, end)` que recorre enteros de `start` a `end` (inclusive), llama a `func_cond(n)` y si es True, aplica `func_accion(n)` y muestra el resultado usando `print()`.
No uses listas; simplemente itera y aplica.

In [ ]:
# Esqueleto
def filtrar_y_aplicar(func_cond, func_accion, start, end):
    # recorrer n desde start hasta end inclusive

    # si func_cond(n) es True -> calcular func_accion(n) y print
    pass

# Prueba sugerida (descomenta):
# filtrar_y_aplicar(lambda x: x%2==0, lambda x: x*x, 1, 10)  # imprime cuadrados de pares

# 3️⃣ Nested functions (Funciones anidadas) y closures

Las funciones anidadas permiten encapsular comportamiento y construir **closures**: funciones que “recuerdan” el entorno en el que fueron creadas. Muy útiles para crear fábricas de funciones o mantener estado privado.

In [ ]:
# Ejemplo: fábrica de multiplicadores (closure)
def make_multiplier(factor):
    def multiply(x):
        return x * factor
    return multiply

mul3 = make_multiplier(3)
print(mul3(5))  # 15

15


### Decoradores (introducción)
Un decorador es una función que toma una función y devuelve una función — es una aplicación práctica de funciones que retornan funciones. Se usan con la sintaxis `@decorator`.

In [ ]:
def mi_decorador(func):
    def wrapper(*args, **kwargs):
        print('Antes de la función')
        resultado = func(*args, **kwargs)
        print('Después de la función')
        return resultado
    return wrapper

@mi_decorador
def di_hola():
    print('Hola!')

di_hola()

Antes de la función
Hola!
Después de la función


### 📝 Ejercicio 4 — Nested functions / Decorador (Esqueleto)
Crea un decorador `contar_llamadas` que añada un atributo `count` a la función decorada y lo incremente cada vez que se llame. Devuelve la función decorada con la propiedad disponible: `f.count`.
Pista: usa closure y `nonlocal` o atributo en la función wrapper.

In [ ]:
# Esqueleto
def contar_llamadas(func):
    # retornar wrapper que incremente contador y llame a func
    pass

# Prueba sugerida:
# @contar_llamadas
# def saludo():
#     print('hola')
# saludo()
# saludo()
# print(saludo.count)  # debe imprimir 2

## ✅ Buenas prácticas y recomendaciones

- **Evita `global`** cuando sea posible; prefiere retornar valores y reasignar externamente.
- Usa **closures** para encapsular estado privado cuando sea útil.
- Documenta funciones con **docstrings** y agrega **type hints** para mayor claridad.
- Para código concurrente o en producción, evita mutable globals y usa clases o estructuras controladas.
- Para depurar scope y closures, usa Python Tutor (pythontutor.com) para visualizar la pila y variables.


## Recursos y lecturas recomendadas
- Documentación oficial — funciones: https://docs.python.org/3/tutorial/controlflow.html#defining-functions
- Artículo sobre closures y scope: https://realpython.com/inner-functions-what-are-closures/
- Python Tutor (visualizador paso a paso): https://pythontutor.com